In [1]:
import re
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

c:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def analyse_dataset(dataset, split="train", sample=5000):
    
    data = dataset[split]
    if len(data) > sample:
        indices = list(range(sample))
        data = data.select(indices)
 
    normals  = data["Normal"]
    simples  = data["Simple"]
 
    n_total       = len(normals)
    n_identical   = 0
    n_simp_longer = 0
    n_too_short   = 0
    n_too_long    = 0
    ratios        = []
    normal_lens   = []
    simple_lens   = []
 
    for norm, simp in zip(normals, simples):
        nl = len(norm.split())
        sl = len(simp.split())
        normal_lens.append(nl)
        simple_lens.append(sl)
 
        if norm.strip().lower() == simp.strip().lower():
            n_identical += 1
        if sl > nl:
            n_simp_longer += 1
        if nl < 5 or sl < 5:
            n_too_short += 1
        if nl > 200 or sl > 200:
            n_too_long += 1
        if nl > 0:
            ratios.append(sl / nl)
 
    print("\n" + "=" * 60)
    print(f"DATASET ANALYSIS — {split} split ({n_total} samples)")
    print("=" * 60)
    print(f"\n  Identical pairs (no simplification) : {n_identical:,}  ({n_identical/n_total*100:.1f}%)")
    print(f"  Simplified LONGER than original      : {n_simp_longer:,}  ({n_simp_longer/n_total*100:.1f}%)")
    print(f"  Too short (< 5 words either side)    : {n_too_short:,}  ({n_too_short/n_total*100:.1f}%)")
    print(f"  Too long  (> 200 words either side)  : {n_too_long:,}  ({n_too_long/n_total*100:.1f}%)")
    print(f"\n  Avg normal length  : {sum(normal_lens)/len(normal_lens):.1f} words")
    print(f"  Avg simple length  : {sum(simple_lens)/len(simple_lens):.1f} words")
    print(f"  Avg length ratio   : {sum(ratios)/len(ratios):.3f}  (1.0 = same length)")
    print(f"  Median ratio       : {sorted(ratios)[len(ratios)//2]:.3f}")
    print("=" * 60 + "\n")
 

In [ ]:
_NOISE_PUNCT_RE = re.compile(r'[\"#$%&\*+/<=>@\[\\\]^_`{|}~]')
_WHITESPACE_RE  = re.compile(r'\s{2,}')
_URL_RE         = re.compile(r'http\S+')
_CITATION_RE    = re.compile(r'\[\d+\]')
_WIKI_TMPL_RE   = re.compile(r'\{\{.*?\}\}')
 
MIN_WORD_COUNT = 5
MAX_WORD_COUNT = 150   
MAX_SIMP_RATIO = 1.3   
 
 
def clean_english(text: str) -> str:
    """Same cleaning pipeline used at inference time in main.py."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = _URL_RE.sub('', text)
    text = _CITATION_RE.sub('', text)
    text = _WIKI_TMPL_RE.sub('', text)
    text = _NOISE_PUNCT_RE.sub('', text)
    text = re.sub(r'\s([,.!?;:])', r'\1', text)
    text = _WHITESPACE_RE.sub(' ', text)
    return text.strip()
 
 
def is_valid_pair(normal: str, simple: str) -> bool:
 
    if not normal or not simple:
        return False
    norm_len = len(normal.split())
    simp_len = len(simple.split())
    if norm_len < MIN_WORD_COUNT or simp_len < MIN_WORD_COUNT:
        return False
    if norm_len > MAX_WORD_COUNT or simp_len > MAX_WORD_COUNT:
        return False
    if normal == simple:           # identical after cleaning
        return False
    if simp_len / norm_len > MAX_SIMP_RATIO:  # simplified is longer
        return False
    return True
 
 
def preprocess_batch(batch):
  
    normals   = []
    simples   = []
 
    for norm, simp in zip(batch["Normal"], batch["Simple"]):
        norm_clean = clean_english(norm)
        simp_clean = clean_english(simp)
        if is_valid_pair(norm_clean, simp_clean):
            normals.append(norm_clean)
            simples.append(simp_clean)
 
    return {
        "normal_clean": normals,
        "simple_clean": simples,
    }
 

In [4]:
MODEL_CHECKPOINT  = "facebook/bart-base"
MAX_INPUT_LENGTH  = 128
MAX_TARGET_LENGTH = 64
 
 
def build_tokenize_fn(tokenizer):
    def tokenize(examples):
        inputs  = ["simplify: " + s for s in examples["normal_clean"]]
        targets = examples["simple_clean"]
 
        model_inputs = tokenizer(
            inputs,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding=False,
        )
        # mask padding with -100 so loss ignores it
        model_inputs["labels"] = [
            [(t if t != tokenizer.pad_token_id else -100) for t in ids]
            for ids in labels["input_ids"]
        ]
        return model_inputs
    return tokenize
 
 

In [5]:

def main():
    print("=" * 60)
    print("BART Simplification — WikiLarge Dataset")
    print("=" * 60)
 
    # ── 1. Load ───────────────────────────────────────────────────────────────
    print("\n[1/6] Loading WikiLarge dataset...")
    dataset = load_dataset("bogdancazan/wikilarge-text-simplification")
    print(f"  Raw train : {len(dataset['train']):,}")
    print(f"  Raw val   : {len(dataset['validation']):,}")
 
    # ── 2. Analyse BEFORE filtering ──────────────────────────────────────────
    # This shows you exactly where the dataset is weak
    print("\n[2/6] Analysing dataset quality...")
    analyse_dataset(dataset, split="train",      sample=5000)
    analyse_dataset(dataset, split="validation", sample=1000)
 
    # ── 3. Preprocess ─────────────────────────────────────────────────────────
    print("[3/6] Preprocessing...")
    dataset = dataset.map(
        preprocess_batch,
        batched=True,
        remove_columns=dataset["train"].column_names,
        desc="Cleaning",
    )
 
    # deduplicate within each split
    for split_name in ("train", "validation"):
        seen = set()
        def dedup(example, seen=seen):
            key = (example["normal_clean"], example["simple_clean"])
            if key in seen: return False
            seen.add(key);  return True
        dataset[split_name] = dataset[split_name].filter(dedup)
 
    print(f"\n  Clean train : {len(dataset['train']):,} samples")
    print(f"  Clean val   : {len(dataset['validation']):,} samples")
    print(f"\n  Sample normal : {dataset['train'][0]['normal_clean']}")
    print(f"  Sample simple : {dataset['train'][0]['simple_clean']}")
 
    # Subset for CPU — remove when moving to GPU
    train_size = min(15000, len(dataset["train"]))
    val_size   = min(500,   len(dataset["validation"]))
    dataset["train"]      = dataset["train"].shuffle(seed=42).select(range(train_size))
    dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(val_size))
    print(f"\n  Using for training   : {len(dataset['train']):,}")
    print(f"  Using for validation : {len(dataset['validation']):,}")
 
    # ── 4. Tokenise ───────────────────────────────────────────────────────────
    print("\n[4/6] Loading tokenizer and tokenising...")
    tokenizer   = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    tokenize_fn = build_tokenize_fn(tokenizer)
 
    remove_cols = [
        c for c in dataset["train"].column_names
        if c not in ("input_ids", "attention_mask", "labels")
    ]
    tokenized = dataset.map(
        tokenize_fn, batched=True,
        remove_columns=remove_cols,
        desc="Tokenising",
    )
    print("  Tokenisation done.")
 
    # ── 5. Model ──────────────────────────────────────────────────────────────
    print("\n[5/6] Loading BART-base...")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
 
    # ── 6. Train ──────────────────────────────────────────────────────────────
    args = Seq2SeqTrainingArguments(
        output_dir                  = "./bart-wikilarge",
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        learning_rate               = 3e-5,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 4,
        num_train_epochs            = 5,
        weight_decay                = 0.01,
        predict_with_generate       = True,
        generation_max_length       = MAX_TARGET_LENGTH,
        fp16                        = False,
        logging_steps               = 50,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        save_total_limit            = 1,
        report_to                   = "none",
    )
 
    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=-100,
    )
 
    trainer = Seq2SeqTrainer(
        model            = model,
        args             = args,
        train_dataset    = tokenized["train"],
        eval_dataset     = tokenized["validation"],
        processing_class = tokenizer,
        data_collator    = data_collator,
        # stop early if eval_loss doesn't improve for 2 epochs
        # saves you hours of unnecessary training on CPU
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
    )
 
    print("\n[6/6] Starting training...")
    trainer.train()
 
    print("\nSaving to ./bart-wikilarge-final ...")
    trainer.save_model("./bart-wikilarge-final")
    tokenizer.save_pretrained("./bart-wikilarge-final")
    print("Done.")
 
 
if __name__ == "__main__":
    main()
 
 

BART Simplification — WikiLarge Dataset

[1/6] Loading WikiLarge dataset...


  Raw train : 148,843
  Raw val   : 494

[2/6] Analysing dataset quality...

DATASET ANALYSIS — train split (5000 samples)

  Identical pairs (no simplification) : 3  (0.1%)
  Simplified LONGER than original      : 1,424  (28.5%)
  Too short (< 5 words either side)    : 0  (0.0%)
  Too long  (> 200 words either side)  : 0  (0.0%)

  Avg normal length  : 23.3 words
  Avg simple length  : 18.6 words
  Avg length ratio   : 0.887  (1.0 = same length)
  Median ratio       : 0.846


DATASET ANALYSIS — validation split (494 samples)

  Identical pairs (no simplification) : 1  (0.2%)
  Simplified LONGER than original      : 143  (28.9%)
  Too short (< 5 words either side)    : 0  (0.0%)
  Too long  (> 200 words either side)  : 0  (0.0%)

  Avg normal length  : 23.3 words
  Avg simple length  : 19.1 words
  Avg length ratio   : 0.899  (1.0 = same length)
  Median ratio       : 0.864

[3/6] Preprocessing...


Filter: 100%|██████████| 438/438 [00:00<00:00, 1625.61 examples/s]



  Clean train : 131,098 samples
  Clean val   : 438 samples

  Sample normal : there is manuscript evidence that austen continued to work on these pieces as late as the period and that her niece and nephew anna and james edward austen made further additions as late as.
  Sample simple : there is some proof that austen continued to work on these pieces later in life. her nephew and niece james edward and anna austen may have made further additions to her work in around.

  Using for training   : 15,000
  Using for validation : 438

[4/6] Loading tokenizer and tokenising...


Tokenising: 100%|██████████| 181/181 [00:00<00:00, 375.71 examples/s]


...

Epoch,Training Loss,Validation Loss
1,1.565631,1.483890
2,1.369240,1.409097
3,1.154442,1.439588
4,1.015100,1.461052


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.16s/it]
C:\Users\fatoo\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it]
C:\Users\fatoo\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]
C:\Users\fatoo\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.86s/


Saving to ./bart-wikilarge-final ...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]


Done.
